### **Submissão 2 — Modelo Transformer**

**Grupo 1 · MIA · Aprendizagem Profunda**

Modelo: Transformer Fine-tuned (BERT / DistilBERT / RoBERTa)
Output: CSV com previsões para o dataset de submissão

In [ ]:
import numpy as np
import pandas as pd
import pickle
import sys, os
import torch
import torch.nn as nn
from transformers import (
    BertTokenizer, BertModel,
    DistilBertTokenizer, DistilBertModel,
    RobertaTokenizer, RobertaModel,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

# 1. Carregar metadados
print('1. A carregar modelo...')
with open('../models/transformer.pkl', 'rb') as f:
    meta = pickle.load(f)

model_name = meta['model_name']
num_classes = meta['num_classes']
class_names = meta['class_names']
max_len = meta['max_len']
freeze_strategy = meta['freeze_strategy']
unfreeze_last_n = meta.get('unfreeze_last_n', 2)
classifier_dropout = meta.get('classifier_dropout', 0.3)

print(f'   Modelo: {model_name}')
print(f'   Freeze: {freeze_strategy}, unfreeze_last_n={unfreeze_last_n}')
print(f'   Classes: {class_names}')

In [ ]:
# 2. Carregar dataset
print('2. A carregar dataset de submissão...')
df = pd.read_csv('../data/subm2.csv', sep=';')
df.columns = df.columns.str.strip().str.lower()
textos = df['text'].tolist()
ids = df['id'].tolist()
print(f'   {len(textos)} textos carregados')

In [ ]:
# 3. Tokenizar
print('3. A tokenizar...')

MODELS = {
    'bert-base-uncased': (BertTokenizer, BertModel, 'bert'),
    'distilbert-base-uncased': (DistilBertTokenizer, DistilBertModel, 'distilbert'),
    'roberta-base': (RobertaTokenizer, RobertaModel, 'roberta'),
}
tok_class, _, _ = MODELS[model_name]
tokenizer = tok_class.from_pretrained(model_name)

enc = tokenizer(textos, add_special_tokens=True, max_length=max_len,
                padding='max_length', truncation=True,
                return_attention_mask=True, return_tensors='pt')
input_ids = enc['input_ids']
attention_mask = enc['attention_mask']
print(f'   Tokens shape: {input_ids.shape}')

In [ ]:
# 4. Classificar
print('4. A classificar...')

class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_classes, freeze_strategy='partial',
                 unfreeze_last_n=2, classifier_dropout=0.3):
        super().__init__()
        self.model_name = model_name
        self.freeze_strategy = freeze_strategy
        _, model_class, self.model_type = MODELS[model_name]
        self.encoder = model_class.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        if freeze_strategy == 'full':
            for param in self.encoder.parameters(): param.requires_grad = False
        elif freeze_strategy == 'partial':
            for param in self.encoder.parameters(): param.requires_grad = False
            if self.model_type == 'bert': layers = self.encoder.encoder.layer
            elif self.model_type == 'distilbert': layers = self.encoder.transformer.layer
            elif self.model_type == 'roberta': layers = self.encoder.encoder.layer
            for layer in layers[-unfreeze_last_n:]:
                for param in layer.parameters(): param.requires_grad = True
        self.classifier = nn.Sequential(
            nn.Dropout(classifier_dropout),
            nn.Linear(hidden_size, 128), nn.ReLU(),
            nn.Dropout(classifier_dropout * 0.5),
            nn.Linear(128, num_classes))
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(outputs.last_hidden_state[:, 0, :])

model = TransformerClassifier(model_name, num_classes,
    freeze_strategy=freeze_strategy, unfreeze_last_n=unfreeze_last_n,
    classifier_dropout=classifier_dropout)
model = model.to(device)
model.load_state_dict(torch.load('../models/transformer.pth',
                                  map_location=device, weights_only=True))
model.eval()

# Classificar em batches
batch_size = 32
all_preds = []
with torch.no_grad():
    for i in range(0, len(textos), batch_size):
        b_ids = input_ids[i:i+batch_size].to(device)
        b_mask = attention_mask[i:i+batch_size].to(device)
        out = model(b_ids, b_mask)
        all_preds.extend(torch.argmax(out, 1).cpu().numpy())

labels_pred = [class_names[i] for i in all_preds]
print(f'   ✅ {len(labels_pred)} previsões feitas')

In [ ]:
# 5. Exportar CSV
print('5. A exportar CSV...')

df_out = pd.DataFrame({'ID': ids, 'Text': textos, 'Labels': labels_pred})

output_path = '../subm2/subm2-g1-MIA-BERT.csv'
os.makedirs('../subm2', exist_ok=True)
df_out.to_csv(output_path, sep=';', index=False, encoding='utf-8')

print(f'✅ Ficheiro guardado: {output_path}')
print(f'\nDistribuição das previsões:')
print(df_out['Labels'].value_counts().to_string())
print(f'\nPrimeiras 10 linhas:')
print(df_out.head(10).to_string(index=False))